In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
idx=pd.IndexSlice

import numpy as np

from datetime import datetime

import itertools

import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
%matplotlib inline

In [ ]:
# import plotting functions for results
#import PlotFunctions as pf


In [ ]:
pd.__version__, np.__version__, sm.__version__

### Define models

In [ ]:
def LMM(IDf, IsMob, IiTimePeriod, IbCorrData, IsCnt, IiModel,
           IbHoly, IbWeekend, IDDfHoly, IiFilterOcc):
   
    # check all values are real
    if not IDf.map(np.isreal).all(0)[IsMob]:
        IDf=IDf[pd.to_numeric(IDf[IsMob], errors='coerce').notnull()]
        IDf=IDf.dropna(subset=[IsMob])
    # levels to numeric values
    IDf=IDf[pd.to_numeric(IDf['Level'], errors='coerce').notnull()]

    # Df for keeping the first day of each occurrence for each level and region
    DfDate=IDf.loc[IDf.groupby(['Region','Level','NOccurrence'])['Date'].idxmin()][['Region','Level','NOccurrence','Date']].copy()
    # drop levels-occurrences lasting less than the chosen duration (14 days in main analysis)
    Dftemp=IDf[['Region','Date','Level','NOccurrence']].copy()
    Dftemp= Dftemp.groupby(['Region','Level','NOccurrence']).count()
    IndexLessPeriod=Dftemp[Dftemp['Date']<IiTimePeriod].index
    if len(IndexLessPeriod)!=0:
        IDf=IDf.set_index(['Region','Level','NOccurrence']).drop(index=IndexLessPeriod).reset_index()
    IDf.dropna(subset=[IsMob], inplace=True)

    # convert data column to float
    IDf.loc[:,IsMob]= IDf.loc[:,IsMob].values.astype(float)

    # drop holidays
    if IbHoly:
        if IsCnt in ['ZAF','SCO','ITA', 'CHL','Ont','CA']:
            IDf=IDf[~IDf['Date'].isin(IDDfHoly[IsCnt]['Date'])]  

    # for each Region, tier and occurrence keep only the first [# number of days: 14 in the main analysis]  
    IDf = IDf[['Region','Level','NOccurrence','Date', sMob,'TotDays','T_Level','DayWeek', 'Incidence']].sort_values(
    ['Region','Level','NOccurrence','Date']).groupby(['Region','Level','NOccurrence']).apply(
        lambda x: x.iloc[:IiTimePeriod].reset_index(drop=True),include_groups=False).reset_index().drop(columns=['level_3'])
        
        
    # add column for calculating the number of days in the year until the occurrence
    IDf['DaySeasonality']=[(Date-datetime(Date.year, 1,1)).days for Date in IDf.Date]  
    if IsCnt=='CHL' or IsCnt=='ZAF':
        lSinDays=[np.sin(2*np.pi/365*(iDate+(365/4-22))+np.pi) for iDate in IDf['DaySeasonality']]
    else:
        lSinDays=[np.sin(2*np.pi/365*(iDate+(365/4-22)))for iDate in IDf['DaySeasonality']]
    # use the new column for calculating the seasonality term
    IDf['Seasonality']= lSinDays

    # drop weekend days
    if IbWeekend:
        IDf=IDf[(IDf['DayWeek']!=5) & (IDf['DayWeek']!=6)]

    # filter on min number of iterations 
    aDropLev= (IDf.groupby('Level').max()['NOccurrence']<IiFilterOcc).loc[lambda x: x==True].index.values
    if len(aDropLev) == len(IDf.Level.drop_duplicates()):
        print ('Warning: dropping all tiers')
    if len(aDropLev) != len(IDf.Level.drop_duplicates()):
        print ('Tiers', aDropLev, ' dropped in ',IsCnt)
        if len(aDropLev) !=0:
            IDf= IDf.set_index('Level').drop(index=aDropLev).reset_index()
        if IbCorrData:
            for l in DaLevels[cnt]:
                # drop tiers where not enough data is avaliable
                if not(IDf[IDf['Level']==l].shape[0]>IDf[IDf['Level']==l].drop_duplicates('Region').shape[0]+1):
                    IDf.drop(IDf.loc[IDf['Level']==l].index, inplace=True)
        # drop entries with missing incidence data
        IDf=IDf.dropna(subset=['Incidence'])

        ## Define statisticl models: from 1 to 7 are Supplementary models S1-S7 , from 8 to 13 are seasonality sensitivity
        ## , from 14 to 16 are the main models and from 17 to 18 are the data filtering sensitivity models##

        ### S1-S7 
        # only Cumulative exposure
        if IiModel== 1:
            eq_Model = smf.mixedlm("{} ~C(Level)+ TotDays  +C(DayWeek) ".format(IsMob), IDf,
                                   groups=IDf['Region'])
            
        # only within-tier duration
        if IiModel== 2:
            eq_Model = smf.mixedlm("{} ~C(Level)+ C(Level):T_Level + C(DayWeek)".format(IsMob), IDf,
                                   groups=IDf['Region'])
            
        # only repeated implementation
        if IiModel==3:
            eq_Model = smf.mixedlm("{} ~  C(Level) + C(Level):NOccurrence +C(DayWeek)".format(IsMob), IDf, 
                                       groups=IDf['Region'])
              
        # Cumulative exposure + within-tier duration 
        if IiModel==4:  
            eq_Model = smf.mixedlm("{} ~C(Level) + TotDays + C(Level):T_Level  +C(DayWeek)".format(IsMob), IDf,
                                  groups=IDf['Region'])
            
        #  Cumulative exposure + repeated implementation
        if IiModel==5:
            eq_Model = smf.mixedlm("{} ~C(Level)+ TotDays + C(Level):NOccurrence +C(DayWeek)".format(IsMob), IDf, 
                                       groups=IDf['Region'])
            
        #  within-tier duration  + repeated implementation
        if IiModel==6:
            eq_Model = smf.mixedlm("{} ~C(Level) +  C(Level):T_Level+ C(Level):NOccurrence+ C(DayWeek)".format(IsMob), IDf, 
                                       groups=IDf['Region'])
             
        # all three temporal dimensions 
        if IiModel==7:
            eq_Model = smf.mixedlm("{} ~ C(Level) + C(Level):NOccurrence + C(Level):T_Level + TotDays + C(DayWeek)".format(IsMob), IDf, groups=IDf['Region'])
        ### end sensitivity models
        
        ### Sensitivity to including Seasonality in the supplementary models: S1Season - S6Season
        if IiModel== 8:
            eq_Model = smf.mixedlm("{} ~C(Level)+ TotDays+ Seasonality  +C(DayWeek) ".format(IsMob), IDf,
                                   groups=IDf['Region'])
            
        # only within-tier duration
        if IiModel== 9:
            eq_Model = smf.mixedlm("{} ~C(Level)+ C(Level):T_Level + Seasonality + C(DayWeek)".format(IsMob), IDf,
                                   groups=IDf['Region'])
            
        # only repeated implementation
        if IiModel==10:
            eq_Model = smf.mixedlm("{} ~  C(Level) + C(Level):NOccurrence + Seasonality +C(DayWeek)".format(IsMob), IDf, 
                                       groups=IDf['Region'])
              
        # Cumulative exposure + within-tier duration 
        if IiModel==11:  
            eq_Model = smf.mixedlm("{} ~C(Level) + TotDays + C(Level):T_Level + Seasonality  +C(DayWeek)".format(IsMob), IDf,
                                  groups=IDf['Region'])
            
        #  Cumulative exposure + repeated implementation
        if IiModel==12:
            eq_Model = smf.mixedlm("{} ~C(Level)+ TotDays + C(Level):NOccurrence+ Seasonality +C(DayWeek)".format(IsMob), IDf, 
                                       groups=IDf['Region'])
            
        #  within-tier duration  + repeated implementation
        if IiModel==13:
            eq_Model = smf.mixedlm("{} ~C(Level) +  C(Level):T_Level+ C(Level):NOccurrence + Seasonality+ C(DayWeek)".format(IsMob), IDf, 
                                       groups=IDf['Region'])
        ### end seasonality sensitivity models

        ### Main models
        # all three temporal dimensions 
        if IiModel==14:
            eq_Model = smf.mixedlm("{} ~ C(Level) + C(Level):NOccurrence + C(Level):T_Level + TotDays + Seasonality + C(DayWeek)".format(IsMob), IDf, groups=IDf['Region'])
            
        # all three temporal dimensions + risk adaptation
        if IiModel == 15:
            eq_Model = smf.mixedlm("{} ~ C(Level) + C(Level):NOccurrence + C(Level):T_Level +\
            TotDays + Seasonality + Incidence +C(DayWeek)".format(IsMob), IDf, groups=IDf['Region'])  
        
        # all three temporal dimensions + risk adaptation + risk habituation
        if IiModel == 16:
            eq_Model = smf.mixedlm("{} ~ C(Level) + C(Level):NOccurrence + C(Level):T_Level +\
            TotDays + Seasonality + Incidence + Incidence:TotDays +C(DayWeek)".format(IsMob), IDf, groups=IDf['Region'])    

        ### Sensitivity data filters
        # senstitivity to duration window
        if IiModel==17:
            eq_Model = smf.mixedlm("{} ~ C(Level) + C(Level):NOccurrence + C(Level):T_Level +\
            TotDays + Seasonality + Incidence + Incidence:TotDays +C(DayWeek)".format(IsMob), IDf, groups=IDf['Region']) 
        # sensitivity to number of iterations
        if IiModel==18:
            eq_Model = smf.mixedlm("{} ~ C(Level) + C(Level):NOccurrence + C(Level):T_Level +\
            TotDays + Seasonality + Incidence + Incidence:TotDays +C(DayWeek)".format(IsMob), IDf, groups=IDf['Region']) 
        # run LMM analysis
        LMM = eq_Model.fit(reml=False)

    else:
        LMM=DfDate

    return (LMM,IDf)



### Load Data

In [ ]:
# store main data
DDfData=dict()
# store data holidays
DDfHoly=dict()

DiLenLevels= dict()
DaLevels= dict()
DlArea= dict()

# dict for associating the color to the tier number 
DColLev={1:'skyblue',2:'gold',3:'darkorange',4:'red',5:'purple',6:'navy'}
lMob=['Workplaces','Residential']
lCnt=['ITA','SCO','Ont','CA','ZAF', 'CHL']
# long names regions
DNames={'ITA':'Italy','SCO':'Scotland','Ont':'Ontario','CA':'California','ZAF':'South Africa', 
        'CHL':'Chile'}

sNameFile='Google'

DsMob=dict(zip(lCnt,[lMob]*len(lCnt)))

print ('You are using the {} mobility data'.format(sNameFile))
for cnt in lCnt[:]:
    print (cnt)
    if cnt=='CHL':
        sNameFile=''
    # load dataframe with holiday dates
    DDfHoly[cnt]=pd.read_csv('../../Data/Mobility_Response/Restrictions/'+cnt+'/'+cnt+'_Holidays.csv',parse_dates=['Date'])

    # load main data fime 
    DDfData[cnt]= pd.read_csv('../../Data/Mobility_Response/Mob'+sNameFile+'_Res_'+cnt+'.csv',parse_dates=['Date'])
    DDfData[cnt]= DDfData[cnt].sort_values(['Region','Date'])
    DDfData[cnt]['Level']= DDfData[cnt]['Level'].astype(int) 
                                    
    # add column with total days spent under NPIs
    days= pd.date_range(start=DDfData[cnt].drop_duplicates('Date').sort_values('Date').iloc[0].Date, end=
            DDfData[cnt].drop_duplicates('Date').sort_values('Date').iloc[-1].Date)
    DDaysCount=dict(zip(days,np.arange(1,len(days)+1)))
    DDfData[cnt]['TotDays']=[DDaysCount[day] for day in  DDfData[cnt].Date]
    
    # add column for the day of the week (0 for Monday and 6 for Sunday)
    DDfData[cnt]['DayWeek']=[day.weekday() for day in DDfData[cnt].Date]

    # add column within-tier days
    for reg in DDfData[cnt].drop_duplicates('Region')['Region']:
        # get the initial and final date of each occurrence
        aIndStart= np.insert(np.where(np.diff(DDfData[cnt].set_index(['Region']).loc[reg]['Level'].values)!=0)[0]+1,0 ,0)
        aIndEnd= np.insert(DDfData[cnt].set_index(['Region']).loc[reg].shape[0]-1, -1,
                           np.where(np.diff(DDfData[cnt].set_index(['Region']).loc[reg]['Level'].values)!=0)[0])

        aStartDate=DDfData[cnt].set_index(['Region']).loc[reg].iloc[aIndStart]['Date'].values
        aEndDate=DDfData[cnt].set_index(['Region']).loc[reg].iloc[aIndEnd]['Date'].values

        # construct dict counting the days in each occurrence (for a region and tier)
        DDaysTLev=dict(zip([i for d in range(aStartDate.shape[0]) for i in pd.date_range(start=aStartDate[d], end=aEndDate[d])],
                 [day for d in range(aStartDate.shape[0]) for day in np.arange(1,
                                        pd.date_range(start=aStartDate[d], end=aEndDate[d]).shape[0]+1)]))
        # cosnstruct column 
        lTLevel=[DDaysTLev[day] for day in DDfData[cnt].loc[DDfData[cnt]['Region']==reg,'Date']]
        DDfData[cnt].loc[DDfData[cnt]['Region']==reg,'T_Level']= lTLevel
    DDfData[cnt]= DDfData[cnt].astype({'T_Level':int})

    
    if cnt=='Ont':
        # drop early stage data for Toronto because it entered later the tiered restrictions system
        DDfData[cnt]=DDfData[cnt].drop(DDfData[cnt][DDfData[cnt]['Color']=='white'].index)                                
            
    if cnt=='CHL':
        DDfData[cnt]= DDfData[cnt].rename(columns={'In':'Residential', 'Out':'Workplaces'})

    DiLenLevels[cnt]= len(DDfData[cnt].drop_duplicates('Level'))
    DaLevels[cnt]= DDfData[cnt].drop_duplicates('Level').sort_values('Level')['Level'].values
    DlArea[cnt]= DDfData[cnt].drop_duplicates('Region')['Region'].values.tolist()

    # add column with occurrences 
    DDfData[cnt]['NOccurrence']=1
    # initiate dict for occurrences in each area
    DOcc= dict()
    for sArea in DlArea[cnt]:
        DOcc[sArea]=dict(zip(DaLevels[cnt],np.ones(DiLenLevels[cnt]).astype(int)))
        DftempArea=DDfData[cnt][DDfData[cnt]['Region']== sArea].sort_values('Date').copy()
        for i in range(1,len(DftempArea)):
            index=DftempArea.iloc[i].name
            itempLevel= DftempArea.iloc[i].Level
            itempLevelBefore= DftempArea.iloc[i-1].Level

            if itempLevel != itempLevelBefore:
                DOcc[sArea][itempLevelBefore]= DOcc[sArea][itempLevelBefore]+1

            DDfData[cnt].loc[index,'NOccurrence']= DOcc[sArea][itempLevel]

        

### Run analysis

In [ ]:
# Define dictinaries for construing the dataframe where saving the ouput of the regressions 

lStats= ['Intercept', 'C(Level)[T.2]', 'C(Level)[T.3]', 'C(Level)[T.4]','C(Level)[T.5]', 'C(Level)[T.6]',
          'C(DayWeek)[T.1]', 'C(DayWeek)[T.2]', 'C(DayWeek)[T.3]', 'C(DayWeek)[T.4]', 'C(DayWeek)[T.5]', 'C(DayWeek)[T.6]',
         'C(Level)[1]:NOccurrence', 'C(Level)[2]:NOccurrence', 'C(Level)[3]:NOccurrence', 
         'C(Level)[4]:NOccurrence', 'C(Level)[5]:NOccurrence','C(Level)[6]:NOccurrence', 
         "C(Level)[1]:T_Level", "C(Level)[2]:T_Level", "C(Level)[3]:T_Level", "C(Level)[4]:T_Level",
         "C(Level)[5]:T_Level", "C(Level)[6]:T_Level",
         'TotDays','Seasonality',
         'Incidence', 'Incidence:TotDays']

lLev=['Global',*list(np.arange(2,7)),*list(np.arange(1,7)),*list(np.arange(1,7)),*list(np.arange(1,7)), 
      *['Global']*4]
DStatsLev= dict(zip(lStats,lLev))

lPara=[*['Level']*6, *['Day of the week']*6, *['Repeated implementation']*6,*['Within-tier duration']*6, 'Cumulative exposure','Seasonality',
        'Risk adaptation', 'Risk habituation']
DStatsPara= dict(zip(lStats,lPara))

lParaType= [*['Intercept']*12,*['Slope']*22]
DStatsParaType= dict(zip(lStats,lParaType))

# initialize dataframe and dictionary for saving the output
lCnt=['ITA', 'SCO', 'Ont', 'CA', 'ZAF', 'CHL']
index = pd.MultiIndex(levels=[[]]*7, codes=[[]]*7, names=['Data','Model','Region','Parameter type',
                                                          'Parameter','Original name', 'Level'])
DfResMod = pd.DataFrame(index=index)


In [ ]:
## statistical models: from 1 to 7 are Supplementary models S1-S7 , from 8 to 13 are seasonality sensitivity
## , from 14 to 16 are the main models and from 17 to 18 are the data filtering sensitivity models ##


## Set global filter quantities
# exclude  data for the tiers where not enough data is available
bCorrData=True
# filter on bank holidays
bHoly=True
# drop weekends
bWeekend=False
# filter on minimum number of occurrences
iFilterOcc= 3
# filter on day-window used 
iTimePeriod=14 
##

sNameFile='Google'
# loop over models
for m, iModel in enumerate(np.arange(1,19)):
    print ('\n','Model: ',iModel)
    # loop over regions
    for cnt in lCnt:
        print ('\n',cnt) 
        for sMob in ['Workplaces', 'Residential']:
            print (sMob)

            Df=DDfData[cnt].copy()
            # drop first tier in Italy
            if cnt=='ITA':
                Df=Df[Df['Level']!=1]

            ## Run model fit
            # Run main and supplementary models
            if iModel<18:
                (tempResMod,Df)=LMM(Df,sMob,iTimePeriod, bCorrData, cnt, iModel,
                                 bHoly, bWeekend, DDfHoly, iFilterOcc)
            else:
                # run the sensitivity models
                # sensitivity on days within a tier
                if iModel==17:
                    iTimePeriod=28
                    bHoly= True
                    iFilterOcc=3
                    (tempResMod,Df)=LMM(Df,sMob,iTimePeriod, bCorrData, cnt, iModel,
                                 bHoly, bWeekend, DDfHoly, iFilterOcc)
                else: 
                    # sensitivity on the number of iterations
                    if iModel==18:
                        iTimePeriod=14
                        bHoly= True
                        iFilterOcc=2
                        (tempResMod,Df)=LMM(Df,sMob,iTimePeriod, bCorrData, cnt, iModel,
                                 bHoly, bWeekend, DDfHoly, iFilterOcc)                    

            print (tempResMod.converged)
            # flag
            if type(tempResMod)==pd.DataFrame:
                print ('Caution, no data for {} mobility in country {}.'.format(sMob,cnt))
                continue
           
            # table model fit
            iLenPara= len(tempResMod.params)-1
            for i in range(iLenPara):

                DfResMod.loc[(sNameFile,'Model ' + str(iModel),DNames[cnt],
                            DStatsParaType[tempResMod.params.index.values[i]],
                            DStatsPara[tempResMod.params.index.values[i]],
                            tempResMod.params.index.values[i],
                            DStatsLev[tempResMod.params.index.values[i]]),
                            'Est. '+sMob]= float(tempResMod.params[i])
                DfResMod.loc[(sNameFile,'Model ' + str(iModel),DNames[cnt], 
                            DStatsParaType[tempResMod.params.index.values[i]],
                            DStatsPara[tempResMod.params.index.values[i]],
                            tempResMod.params.index.values[i],
                            DStatsLev[tempResMod.params.index.values[i]]),
                            'CI95% '+sMob]= float(tempResMod.params[i])-\
                            float(tempResMod.conf_int()[0][i])

            # calculate marginal and conditional R2
            dVarResiduals= tempResMod.resid.var()
            dVarRE=tempResMod.cov_re.Group.values[0]
            dVarFE= tempResMod.predict().var()
            dmR2=round(dVarFE/ (dVarFE + dVarRE + dVarResiduals), 2)
            dcR2= round((dVarFE+ dVarRE)/(dVarFE+dVarRE+dVarResiduals), 2)
                       
            DfResMod.loc[(sNameFile,'Model '+str(iModel),DNames[cnt], 'mR2', 'All','All','All'), 'Est. '+sMob]= dmR2
            DfResMod.loc[(sNameFile,'Model '+str(iModel),DNames[cnt], 'cR2', 'All','All','All'), 'Est. '+sMob]= dcR2
            DfResMod.loc[(sNameFile,'Model '+str(iModel),DNames[cnt], 'AIC', 'All','All','All'), 'Est. '+sMob]= tempResMod.aic

DfResMod.columns=[[*list(itertools.chain(*[[mob]*2 for mob in ['Workplaces', 'Residential']]))],
              DfResMod.columns.values.tolist()]

DfResMod.sort_index(inplace=True)
